In [1]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import DistilBertTokenizerFast, get_linear_schedule_with_warmup, BertForTokenClassification
from optional_fine_tune import BertTokenClassification, DfToDataset, train_epoch, eval_model

D:\Codding\Education\NLP\Tweet Sentiment Extraction\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("data/train.csv")
train = train.dropna().reset_index(drop=True)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

Используем устройство: cuda


In [4]:
MAX_LEN = 96
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 3e-5
N_SPLITS = 5

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_jaccard_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X=train, y=train['sentiment'])):
    train_section = train.iloc[train_idx].reset_index(drop=True)
    val_section = train.iloc[val_idx].reset_index(drop=True)

    train_dataset = DfToDataset(train_section, tokenizer, max_len=MAX_LEN)
    val_dataset = DfToDataset(val_section, tokenizer, max_len=MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False)

    model = BertTokenClassification().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    best_jaccard = 0.0

    for epoch in range(EPOCHS):
        print(f"Эпоха {epoch + 1}/{EPOCHS}")

        train_loss = train_epoch(model, optimizer, train_loader, loss_fn, scheduler, device)
        val_jaccard = eval_model(model, val_loader, device)

        print(f"Loss обучения: {train_loss:.4f} | Jaccard валидации: {val_jaccard:.4f}")

        if val_jaccard > best_jaccard:
            best_jaccard = val_jaccard
            torch.save(model.state_dict(), f"models/best_bert_fold_{fold + 1}.pt")
            print("--> Найдена лучшая модель, веса сохранены!")

    fold_jaccard_scores.append(best_jaccard)
    print(f"Лучший Jaccard на Фолде {fold + 1}: {best_jaccard:.4f}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9139.91it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3
Loss обучения: 1.3673 | Jaccard валидации: 0.6891
--> Найдена лучшая модель, веса сохранены!
Эпоха 2/3
Loss обучения: 0.7934 | Jaccard валидации: 0.6966
--> Найдена лучшая модель, веса сохранены!
Эпоха 3/3
Loss обучения: 0.6758 | Jaccard валидации: 0.7014
--> Найдена лучшая модель, веса сохранены!
Лучший Jaccard на Фолде 1: 0.7014


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9191.58it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3
Loss обучения: 1.3489 | Jaccard валидации: 0.6986
--> Найдена лучшая модель, веса сохранены!
Эпоха 2/3
Loss обучения: 0.7968 | Jaccard валидации: 0.7072
--> Найдена лучшая модель, веса сохранены!
Эпоха 3/3
Loss обучения: 0.6818 | Jaccard валидации: 0.7073
--> Найдена лучшая модель, веса сохранены!
Лучший Jaccard на Фолде 2: 0.7073


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7968.05it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3
Loss обучения: 1.3482 | Jaccard валидации: 0.6906
--> Найдена лучшая модель, веса сохранены!
Эпоха 2/3
Loss обучения: 0.7937 | Jaccard валидации: 0.6995
--> Найдена лучшая модель, веса сохранены!
Эпоха 3/3
Loss обучения: 0.6771 | Jaccard валидации: 0.7035
--> Найдена лучшая модель, веса сохранены!
Лучший Jaccard на Фолде 3: 0.7035


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10161.85it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3
Loss обучения: 1.3417 | Jaccard валидации: 0.7078
--> Найдена лучшая модель, веса сохранены!
Эпоха 2/3
Loss обучения: 0.7900 | Jaccard валидации: 0.7112
--> Найдена лучшая модель, веса сохранены!
Эпоха 3/3
Loss обучения: 0.6737 | Jaccard валидации: 0.7122
--> Найдена лучшая модель, веса сохранены!
Лучший Jaccard на Фолде 4: 0.7122


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11025.17it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3
Loss обучения: 1.3587 | Jaccard валидации: 0.6963
--> Найдена лучшая модель, веса сохранены!
Эпоха 2/3
Loss обучения: 0.7846 | Jaccard валидации: 0.6979
--> Найдена лучшая модель, веса сохранены!
Эпоха 3/3
Loss обучения: 0.6721 | Jaccard валидации: 0.7030
--> Найдена лучшая модель, веса сохранены!
Лучший Jaccard на Фолде 5: 0.7030


In [6]:
print(f"КРОСС-ВАЛИДАЦИЯ ЗАВЕРШЕНА")
print(f"СРЕДНИЙ JACCARD (CV Score): {np.mean(fold_jaccard_scores):.4f}")

КРОСС-ВАЛИДАЦИЯ ЗАВЕРШЕНА
СРЕДНИЙ JACCARD (CV Score): 0.7055
